# **RESDES NEURONALES - LSTM**

* La defensa y dominio del tema no debe limitarse a **"como usar la libreria"**, sino a entender **la dinamica matematica y el fujo de gradientes** dentro de la red.

* Nuestro tema a investigar es el **Cetelleo Ionosférico**(una serie de tiempo caótica y afectada por variables fisicas), las LSTM son una elección canónica y robusta.

* Cuando decimos que las LSTM son una **"elección canonica"** para series de tiempo, estamos haciendo referencia a varias cosas:

    1. **Estándar de facto:** Es una de la primeras **arquitecturas** en la que se piensa y se prueba cuando se aborda un problema de series temporales  con dependencia a largo plazo. Las redes LSTM se han convertido en un "caballo de batalla" probado y validado en la investifacion y la industria para esta clase de problemas.

    2. **Manejo intrinseco de dependencias temporales:** A diferencia de las redes feedforwad o incluso algunos modelos estadisticos, las LSTM estan diseñadas desde su concepción para capturar y recordar patrones a lo largo del tiempo. Su *cell state*  y sus compuerta permiten manejar tanto la "memoria a corto pazo"(a través del hidden state) como la "memoria a largo plazo" (a través del cell state) de manera efectiva, resolviendo el problema del desvanecimiento del gradiente.
    3. **Robustez en datos ruidosos y no lineales:** Las series de tiempo, como las de Centelleo inosfericos, suelen ser ruiododas, no estacionarias y presentan relaciones lineales complejas entre las variables. Las LSTM, siendo redes neuronales profundas, son inheremente buenas para modelar estas complejidades sin necesidad de una ingenieria de caracteristicas manual exhaustiva(aunque el preprocesamiento siempre ayuda).

    4. **Capacidad para multiples entradas(multivariado):** Como veremos en nuestro caso las LSTM pueden manejar facilmente multiples series de tiempo correlacionadas(S4,TEC, ROTI,SW,etc) como entradas simultaneas para haceruna predicción.

    5. **Gran cantidad de investigaciones y soporte:** Hay una amplia literatura, frameworks de deep learning(TensorFlow, Pytorch) y una comunidad activava que respalda el desarrollo y uso de las LSTM, lo que facilita su implementación y solución de problemas.

# **Aquitectura Propuesta: Modelo Hibrido Morph-LST-ELM**

## **1 Descripcion General del Sistema**

Para abordar la naturaleza no lineal, caotica y ruidosa  de las series temporales de centelleo ionosferico(Indice S4), esta investigacion propone una arquitectura hibrida profunda denominada *Morph-LSTM-ELM*. El modelo integra técnicas de procesamiento de señales no lineales(Morfología Matemática) con redes neuronales recurrentes profundas(Deep LSTM) y algortimos de aprendizaje extremo(ELM)

El flujo de procesamiento se divide en tres etapas secuenciales:
1. **Filtrado Estructural:** Reducción de ruido estocástico mediantes operadores morfológicos.
2. **Extracción de Características Temporales:** Modelado de dependencias a largo plazo mediante capas LSTM.
3. **Inferencia Analítica:** Regresión final mediante la solución de mínimos cuadrados(ELM), sustituyendo el descenso de gradiente en la capa de salida.



## **2. Pre-Procesamiento: Filtrado Morfológico 1D**

Dada la alta sensibilidad de los receptores GNSS, la serie temporal del indica S4 presenta componentes de ruido de alta frecuencia que no corresponden a eventos fisicos de centelleo, sino a perturbaciones instrumentales o multipath. Para mitigar esto sin alterar la fase de los picos de centelleo, se aplica un filtro morfológico de Apertura(Opening) sobre la serie univariada S4.

Sea f(t) la señal original de  $S_4$  y B un elemento estructurante plano de longitud lambda. La operación de apertura $\gamma_B(f)$ se define como la erosión seguida de la dilatación:

$$\gamma_B(f) = (f \ominus B) \oplus B$$

Donde:
* Erosión ($\ominus$): $(f \ominus B)(t) = \min_{s \in B} \{ f(t+s) \}$
* Dilatación ($\oplus$): $(f \oplus B)(t) = \max_{s \in B} \{ f(t-s) \}$

Este proceso suaviza el contorno de la señal eliminando picos positivos espurios que tienen una duración menor al elemento estructurante B, preservando la "envolvente" de los eventos de centelleo reales para el entrenamiento.

## **3. Arquitectura del Núcleo: LSTM-ELM**

La arquitectura propuesta sustituye la capa densa(Fully Connected) tradicional de una red LSTM estándar por un mecanismo de Extreme Learning Machine(ELM) para mejorar la capacidad de generalización y la velocidad de convergencia.

### **3.1 Extractor de Característica(Backbone LSTM)**

El nucleo del modelo recibe un tensor de entrada $\mathbf{X} \in \mathbb{R}^{N \times T \times D}$, donde T es la ventana de tiempo histórica y D es el número de variables físicas (incluyendo S4, filtrado, TEC,ROTI, indices geomagnéticos $K_p, D_{st}$, y viento solar)
Las capas LSTM procesan la secuencia secuencialmente para generar un vector de estado oculto ht que encapsula la memoria del sistema.

$$h_t = \text{LSTM}(\mathbf{x}_t, h_{t-1}, C_{t-1})$$

La salida final de la ultima capa recurrente no se utiliza para prediccion directa, sino que constituye una matriz de característica latentes H, que representa la dinámica compleja de la ionósfera de un espacio dimensional superior.

## **3.2 Regresor Analítico(Capa ELM)**

A diferencia de la redes convencionales que ajustan los pesos de la capa de salida ($W_{out}$) iterativamente mediante Backpropagation, este modelo propone calcular $W_{out}$ analíticamente . Se formula el problema como un sistema lineal:

$$\mathbf{H} \cdot \beta = \mathbf{T}$$

Donde:
* H: Matriz de salida de la capa LSTM(Hidden Layer Output Matrix)
* $\beta$: Matriz de pesos de salida a determinar.
* $\mathbf{T}$: Vector de objetivos (Valores reales de $S_4$ a predecir).

La solución óptima para $\beta$, que minimiza la norma del error $\| \mathbf{H}\beta - \mathbf{T} \|$, se obtiene mediante la Pseudoinversa de Moore-Penrose ($\mathbf{H}^{\dagger}$):

$$\hat{\beta} = \mathbf{H}^{\dagger} \mathbf{T}$$

Esta aproximación garantiza el alcance del óptimo global para la capa de salida y evita el estancamiento en mínimos locales comunes en el entrenamiento estocástico.

## **4. Estrategia de Entrenamiento y Función de Costo**
El entrenamiento se realiza en dos fases("Two-stage training"):
1. **Fase de Representacion:** Se entrena la red LSTM completa(con una capa temporal de salida) utilizando el algoritmo Adam y Backpropagation Through Time(BPTT). El objetivo es que los pesos internos ($W_f, W_i, W_C, W_o$) aprendan a extraer características relevantes del clima espacial.

2. **Fase de Regresión:** Se congelan los pesos de la LSTM. Se propongan los datos de entrenamiento para obtener la matriz H y se calculan $\beta$ mediante la ecuación de la pseudoinversa.

## **4.1 Funcion de Perdida Ponderada(Weighted Loss)**
Para la evaluación y ajuste fino, se de fine una función de pérdida asimétrica que penalza severamente los errores en eventos de centelleo intenso($S_4 > \tau$), criticos para las telecomunicaciones:

$$L(\mathbf{y}, \hat{\mathbf{y}}) = \frac{1}{N} \sum_{i=1}^{N} w_i (y_i - \hat{y}_i)^2$$

$$w_i = \begin{cases} \alpha & \text{si } y_i \ge \tau \quad (\text{Evento de Centelleo}) \\ 1 & \text{si } y_i < \tau \quad (\text{Calma Ionosférica}) \end{cases}$$

Donde $\tau$ es el umbral de disturbio y $\alpha > 1$ es el factor de penalización, asegurando que el modelo priorice la sensibilidad (detección de picos) sobre la especificidad en zonas de calma.

## **DIAGRAMA ARQUITECTONICO**
Arquitectura del Sistema Morph-LSTM-ELM no como una "caja negra", sino como un flujo de tensores.
* Entrada: Tensor $\mathbf{X} \in \mathbb{R}^{Batch \times Steps \times Features}$
* Bloque 1: Morphological Filter (1D) $\rightarrow$ Tensor Limpio $\mathbf{X}'$
* Bloque 2: Stacked LSTM Layers (Feature Extraction) $\rightarrow$ Matriz Latente $\mathbf{H} \in \mathbb{R}^{Batch \times Units}$
* Bloque 3: ELM Solver (Linear Projection) $\rightarrow$ Pseudoinversa $\mathbf{H}^{\dagger}$
* Salida: Escalar $\hat{y}_{t+k}$ (Pronóstico $S_4$)

# **EXPLICACION OBJETIVA  PARA DECIDIR CON FUNDAMENTE**

## **1 ¿QUE ES EL FILTRADO MORFOLÓGICO?**

Imaginemos que tenemos una carretera llena de piedras pequeñas(ruido) y grandes rocas(el evento que te interesa).
* Un promedio móvil(filtro clásico) es como pasar una aplanadora: aplasta las piedras y las rocas por igual, dejando todo "suave" pero deformando las rocas grandes.
* El **filtrado morfológico** es como pasar un tamiz o una rejilla de cierto tamaño. Si la piedra es mas pequeña que la rejilla, desaparece. Si es una roca grande, se queda intacta, conservando sus bordes afilados.

En términos técnicos: Es una técnica lineal que analiza la forma(geometria) de la señal usando una "ventana" llamado **Elemento Estructurante**. A diferencia de filtros de frecuencia(como Fourier o Butterworth), la morfologia no se preocupa por los Hz, sino por la topología de los picos y  valles.

## **2 ¿Como funciona en tu S4?**
El indice S4 es una señal "sucia".Tiene picos reales(tormenta ionosférica) y picos falsos(ruido del receptor, multipath y perdida de ciclo)

1. **Erosion:** "Come" los bordes de la señal. Elimina picos estrechos.

2. **Dilatacion:** "Infla" la señal. Rellena huecos.
Para tu tesis, la operación reina es la **APERTURA(OPENING)**
* **Fórmula:** Elimina los picos positivos que son mas delgados que tu ventan de tiempo(ruido), pero **respeta la altura y forma** de los picos anchos(eventos de centelleo real)
* **Diferencia Clave:** Un filtro promedio haria que el pico de centello parezca mas bajo y mas ancho de lo que es. La morfología mantiene la altura real del pico(importante para saber la severidad de la tormenta)



## **¿3. Como se utiliza?**
Es relativamente simple implementar esto desde python utilizando la libreria scipy. No se necesita programar la matematica desde cero.


In [ ]:
import numpy as np 
import scipy.ndimage as ndimage
import matplotlib.pyplot as pyplot
# Supongamos que 'S4_RAW' es tu array de datos ruidosos
# window_size: El tamaño del "Elemento estructurante"
# Si los datos son por minuto, un tamaño de 5 significa "Elimina picos que duren menos de window size=5"
window_size = 5

# Aplicamos APERTURA(Opening):Erosion -> Dilatación
# Esto borra el ruido impulsivo(picos falsos)
s4_clean = ndimage.grey_opening(s4_raw, size=(window_size,))

#OPCIONAL: Si se tiene huecos hacia abajo (dropouts), usamos cierre(Closing)
# s4_clean = ndimage.grey_closing(s4_clean, size = (window_size,))

# Visualizacion para el trabajo
plt.plot(s4_raw, label = "S4 Crudo (ruidoso)",alpha=0.5)
plt.plot(s4_clean, label = "S4 Filtrado Morfológico",color="red")
plt.legend()
plt.show()